# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [55]:
# TO recalculate: 
# remove token_df file
# remove enhanced trades
# remove processed trades
# remove processed markets

# rm /Users/vobornij/projects/polymarket/data/markets_processed/*.parquet
# rm /Users/vobornij/projects/polymarket/data/trades_polygon_enriched/enriched_*
# rm /Users/vobornij/projects/polymarket/data/polygon_trades_processed/*.parquet

In [56]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [57]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2027, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 4, 15)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

The token lookup table (`token_id → condition_id, outcome, token_winner, final_price`) is
built once via `build_token_lookup_df()` and persisted to
`data/markets_processed/token_lookup.parquet`. Subsequent runs load it with
`load_token_lookup_df()` instead of re-parsing all market JSON.

In [58]:
# # Load only the market files whose month falls within [START_DATE, END_DATE].
# import datetime as _dt
# def _file_in_range(p, start, end):
#     """Return True if YYYY-MM.parquet filename falls within the date range."""
#     try:
#         y, m = (int(x) for x in p.stem.split("-"))
#         file_date = _dt.date(y, m, 1)
#         return start.replace(day=1) <= file_date <= end.replace(day=1)
#     except Exception:
#         return False

# market_files = [
#     p for p in sorted(MARKETS_DIR.glob("*.parquet"))
#     if _file_in_range(p, START_DATE, END_DATE)
# ]
# print(f"Found {len(market_files)} market file(s)")

# all_market_rows = pd.concat(
#     [pd.read_parquet(f) for f in market_files],
#     ignore_index=True,
# )
# # deduplicate by condition_id (keep first occurrence)
# all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
# print(f"Unique markets (all):  {len(all_market_rows):,}")

# # ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# # Parse end_date_iso as a date; keep only markets whose resolution date falls
# # within [START_DATE, END_DATE] (inclusive).
# end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
# in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
# all_market_rows = all_market_rows[in_range].reset_index(drop=True)
# print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
# all_market_rows.head(2)

In [59]:
from polymarket_analysis.data.data_catalogue import (
    load_markets_processed,
    build_token_lookup_df,
    load_token_lookup_df,
)
token_df = load_token_lookup_df()
print(f"Token lookup rows: {len(token_df):,}")

token_df = token_df[
    ~(token_df['primary_tag'].isin(['Sports', 'Crypto']))
    & token_df['token_winner'].notna()
    ]
len(token_df)

Processing 74 market file(s)…
Saved 2,286,384 token rows → /Users/vobornij/projects/polymarket/data/markets_processed/token_lookup.parquet
Token lookup rows: 2,286,384


314388

## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel.  Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal.  The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

In [60]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

trade_files = sorted(TRADES_DIR.glob("*.parquet"))

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS, token_df): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

4717586 trades in 1.parquet
4932042 trades in 0.parquet
4783918 trades in 2.parquet
3576828 trades after merging with token_df for 1.parquet
5117319 trades in 3.parquet
3358063 trades after merging with token_df for 0.parquet
3959307 trades after merging with token_df for 2.parquet
5420962 trades in 4.parquet
4110058 trades after merging with token_df for 3.parquet
4934142 trades in 5.parquet
4347466 trades after merging with token_df for 4.parquet
5293802 trades in 7.parquet
3738903 trades after merging with token_df for 5.parquet
5445483 trades in 6.parquet
4942684 trades in 8.parquet
4613374 trades in 9.parquet
4203119 trades after merging with token_df for 7.parquet
4525720 trades after merging with token_df for 6.parquet
3956999 trades after merging with token_df for 8.parquet
3547769 trades after merging with token_df for 9.parquet
Enriched 0.parquet with copyable_qty
Enriched 1.parquet with copyable_qty
5854327 trades in a.parquet
4382228 trades after merging with token_df for a

In [61]:
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', 200)

# MARKET = '0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b'

# df = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
# df = df[df['condition_id'] == MARKET]
# df[['side', 'price', 'quantity', 'ts', 'token_id', 'tx_hash', 'wallet', 'position']].head(5)

In [62]:
# len(df)
# df[df['tx_hash']=='0xffece5c5cde2b0e1b9375cf30cbb3af09f087967143aff3b4fe5e53c8d1b58c3']

In [63]:
# df['ts'] = pd.to_datetime(df['block_timestamp'], unit='s')
# df['token_id'] = df['token_id'].str[:5]
# df['tx_hash'] = df['tx_hash'].str[:5]
# df[["wallet", 'tx_hash', 'ts', "side", "price", "quantity","token_id", "token_winner", "avail_copy_qty", "avail_copy_total_vol", "avail_copy_count","condition_id"]].head(1)

In [64]:
from polymarket_analysis.preprocessing.shard_processor import select_top_wallets_shard

# ── load token resolution DataFrame ─────────────────────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard...")
shard_wallet_pnls: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
shard_wallet_thresholds: dict[int, float] = {}   # shard index → top-pct threshold (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(select_top_wallets_shard, f, token_df, END_TRAIN_TS, top_pct=0.04): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnls or pnl > shard_wallet_pnls[w]:
                shard_wallet_pnls[w] = pnl
        shard_wallet_thresholds[i] = stats["threshold"]
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnls):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnls.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")
print(f"Threshold pnls per shard: {shard_wallet_thresholds}")

Phase 1 — selecting top-4% wallets per shard...
Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...
Shard top 4.00% threshold: 25.46 USDC
Processing shard enriched_2.parquet...
Shard top 4.00% threshold: 27.06 USDC
Processing shard enriched_3.parquet...
Shard top 4.00% threshold: 36.21 USDC
Processing shard enriched_4.parquet...
Shard top 4.00% threshold: 71.13 USDC
Processing shard enriched_5.parquet...
Shard top 4.00% threshold: 52.79 USDC
Processing shard enriched_6.parquet...
Shard top 4.00% threshold: 24.72 USDC
Processing shard enriched_7.parquet...
Processing shard enriched_8.parquet...
Shard top 4.00% threshold: 50.96 USDC
  4/16 shards | raw: 15,774,894 | in-range: 15,774,894 | wallet union so far: 28,062
Processing shard enriched_9.parquet...
Shard top 4.00% threshold: 42.67 USDC
  8/16 shards | raw: 31,819,464 | in-range: 31,819,464 | wallet union so far: 44,949
Shard top 4.00% threshold: 42.42 USDC
Processing shard enriched_a.parquet...
Shard top 4

## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [65]:
from polymarket_analysis.preprocessing.shard_processor import enrich_and_group_shard

# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training P&L per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(enrich_and_group_shard, f, token_df, END_TRAIN_TS, top_wallets): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
# grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)

Phase 2 — grouping and filtering shards...
  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 26,814,440
  Unique wallets in grouped:                   69,296


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,price_x_qty_sum,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0x9fb5ad77b60ada07898f68d1d1bd959f924789b918421cd3284ae41599881a3c,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 06:27:41+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,10435.00,10435.00,20.87,20.87,0.00,2,-20.87,0.00,-0.00,-0.00,0.00,0.00
1,0xf9c6f7aa5883f574563ec4ba53fa49f5b1fcd43cb86a09fa06b496a2c5672e4e,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 07:25:19+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,20870.00,10435.00,20.87,20.87,0.00,2,-20.87,-0.00,0.00,-0.00,0.00,0.00
2,0x31d4022f503889f8acdbbb0f93b4db52fa80b454deca1026bc0dadb4af48aa80,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:03+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,18007.01,2862.99,5.73,5.73,0.00,1,5.73,5.73,2862.99,8.65,2.00,0.00
3,0x04705dba811ad7cbc99d7482b2f9b71319117c66ea7535d7b6b33d1869e919f6,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:53+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,10435.00,7572.01,15.14,15.14,0.00,1,15.14,15.14,7572.01,28.69,3.00,0.00
4,0xfb8fd88523629a26116f05c1f72c7a36e6aa386640d29cf5f84d87171df212c9,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 09:11:33+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,20650.00,10215.00,20.43,20.43,0.00,2,-20.43,-0.00,0.00,-0.00,0.00,0.00


## 4 · Phase 3: apply final PnL cut-off and export

Compute the true cross-shard training P&L per wallet.  Use the **median** of the
per-shard P&L values accumulated in Phase 1 as the export cut-off: wallets whose
total training P&L falls below that median are dropped before writing.

In [66]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",        "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        copyable_pnl       = ("copyable_pnl",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
    )
    .sort_values("copyable_pnl", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard pnl values collected in Phase 1 ─────
import statistics
pnl_cutoff = statistics.median(shard_wallet_thresholds.values())

print(f"\nUnique wallets in training data: {wallet_summary['wallet'].nunique():,}")
print(f"wallet_summary['copyable_pnl'] quantiles:\n{wallet_summary['copyable_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}")

final_wallets = set(
    wallet_summary[
        (wallet_summary["copyable_pnl"] >= pnl_cutoff) &
        (wallet_summary["trade_pnl"] >= wallet_summary["copyable_pnl"])
    ]["wallet"]
)

print(f"\nFinal selected wallets (Phase 2 filter): {len(final_wallets):,}")
print(f"final_wallets['copyable_pnl'] quantiles:\n{wallet_summary[wallet_summary['wallet'].isin(final_wallets)]['copyable_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}")

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard PnL median (cut-off):        {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing PnL cut-off:           {len(final_wallets):,}")
wallet_summary.head(5)


Unique wallets in training data: 69,296
wallet_summary['copyable_pnl'] quantiles:
0.01   -9919.36
0.10    -642.40
0.20    -138.72
0.30      -4.42
0.40      37.01
0.50      60.49
0.60      92.40
0.70     149.61
0.80     281.22
0.90     740.13
0.99   10627.97
Name: copyable_pnl, dtype: float64

Final selected wallets (Phase 2 filter): 18,421
final_wallets['copyable_pnl'] quantiles:
0.01      36.15
0.10      49.99
0.20      66.68
0.30      88.19
0.40     118.20
0.50     164.43
0.60     244.92
0.70     396.88
0.80     736.57
0.90    1905.35
0.99   26855.59
Name: copyable_pnl, dtype: float64
Unique wallets after Phase 2 union:    69,296
Per-shard PnL median (cut-off):        34.55 USDC
Wallets passing PnL cut-off:           18,421


,wallet,num_markets,num_trades,total_cost_usdc,copyable_pnl,trade_pnl
0,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,42,2385,3878059.88,501531.92,1304494.87
1,0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,605,51697,18229909.25,443561.76,1439468.81
2,0x88c4919de76e526d55a32c1f8afb439dd1f1129a,93,7522,1957742.48,357782.51,677687.54
3,0x1caa6a7ad0c6916aef7b67946de2e57ad24846a0,10,1727,213660.22,327426.01,383603.10
4,0xde7be6d489bce070a959e0cb813128ae659b5f4b,104,7327,4798126.76,320549.64,776109.54


## 5 · Market-level volume summary

In [67]:
# join market metadata (question, end_date) for display
markets_meta = load_markets_processed()[["condition_id", "question", "end_date_iso"]].rename(
    columns={"end_date_iso": "end_date"}
)
grouped_meta = grouped.merge(
    markets_meta,
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Processing 74 market file(s)…
Saved 1,238,946 markets → /Users/vobornij/projects/polymarket/data/markets_processed/markets.parquet
Unique markets: 63,914


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69dccbde93147ee4443246,US forces enter Iran by April 30?,2026-04-30T00:00:00Z,6737,204523,218784644.42,217475672.02
1,0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,US x Iran ceasefire by April 7?,2026-04-07T00:00:00Z,5714,162037,143055367.52,148462572.91
2,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5,US government shutdown Saturday?,2026-01-31T00:00:00Z,6548,264812,135210849.55,138549678.83
3,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,Will Zelenskyy wear a suit before July?,2025-06-30T00:00:00Z,2465,76185,128797412.32,129196361.01
4,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd,"US x Iran ceasefire extended by April 22, 2026?",2026-04-21T00:00:00Z,2028,56788,115975323.43,115674653.63
5,0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad17755f9cee123c7099b35,"Russia x Ukraine ceasefire by May 31, 2026?",2026-05-31T00:00:00Z,1382,36356,106614673.37,106013272.87
6,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Khamenei out as Supreme Leader of Iran by February 28?,2026-02-28T00:00:00Z,7621,218197,106243552.66,116255439.18
7,0x7cb525e831729325d651017f81cbcb6f1adde5011c7b2283babea00b4ae93ae7,Netanyahu out by March 31?,2026-03-31T00:00:00Z,3524,124660,95789213.34,96593064.92
8,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,"US strikes Iran by February 28, 2026?",2026-01-31T00:00:00Z,9350,222219,75185571.13,92236683.52
9,0xef8cf8b45ef7e3a607a72b6e1d56bede869fdd81795b63db847de1090bf11c41,TikTok banned in the US before May 2025?,2025-04-30T00:00:00Z,1592,42416,73051019.06,73765153.08


## 6 · Wallet statistics (quantiles)

In [68]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"]    = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,69,1.00,1.00,-61189.72,-111014.20,69227
0.01,693,1.00,1.00,-9919.36,-16538.34,68603
0.05,3465,3.00,1.00,-1784.02,-2863.03,65831
0.10,6930,6.00,1.00,-642.40,-1056.96,62366
0.25,17324,16.00,3.00,-53.81,-164.10,51972
0.50,34648,50.00,8.00,60.49,23.77,34648
0.75,51972,169.00,24.00,199.21,249.99,17324
0.90,62366,531.00,64.00,740.13,1339.63,6930
0.95,65831,1187.20,120.00,1734.99,3879.73,3465


## 7 · Full enriched trade table

In [69]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0x9fb5ad77b60ada07898f68d1d1bd959f924789b918421cd3284ae41599881a3c,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 06:27:41+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,10435.00,10435.00,0.00,False,0.00,20.87,0.00,-20.87,0.00,2
1,0xf9c6f7aa5883f574563ec4ba53fa49f5b1fcd43cb86a09fa06b496a2c5672e4e,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 07:25:19+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,20870.00,10435.00,0.00,False,0.00,20.87,0.00,-20.87,-0.00,2
2,0x31d4022f503889f8acdbbb0f93b4db52fa80b454deca1026bc0dadb4af48aa80,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:03+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,18007.01,2862.99,0.00,False,0.00,5.73,0.00,5.73,5.73,1
3,0x04705dba811ad7cbc99d7482b2f9b71319117c66ea7535d7b6b33d1869e919f6,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:53+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,10435.00,7572.01,0.00,False,0.00,15.14,0.00,15.14,15.14,1
4,0xfb8fd88523629a26116f05c1f72c7a36e6aa386640d29cf5f84d87171df212c9,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 09:11:33+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,20650.00,10215.00,0.00,False,0.00,20.43,0.00,-20.43,-0.00,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26814435,0x3bcd973a3ee5617ac8b063536a42ebc524c9863ceaab5833c57501df536833ca,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-08 18:51:25+00:00,0xc734f61dbb7fbf5ff7afac8c18b1f05f13c4b9b408f8f8d6d210a862ebabd00d,Yes,2500.00,1000.00,0.01,False,0.00,6.00,0.00,-6.00,-6.00,1
26814436,0x070c8061d33e893ea6c921e020c1fc3bd318e437687c5e19be1c3ea3b211eddd,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 20:24:18+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,200.00,200.00,0.02,False,0.00,4.00,0.00,-4.00,-4.00,1
26814437,0x852015bc27fa54364ba4bd307b710982f8c475e33ff5e2ff1c46933b30c74547,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 23:00:20+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,4200.00,4000.00,0.00,False,0.00,12.00,0.00,-12.00,-12.00,1
26814438,0x8815e825e712da69beadc37cf8ddc48f2eb2ce9eaec7c859f3fc1bd1ce740b6f,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 16:39:01+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Yes,530.00,50.00,0.51,True,1.00,25.35,50.00,-24.65,-24.65,2


## 8 · Export: apply PnL cut-off and write partitioned parquet

Filter grouped trades to ``final_wallets`` (wallets above the median per-shard PnL),
then write one parquet shard per first hex character of ``condition_id`` after ``0x``.

In [70]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 18,421


In [71]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  5,647,950
  training rows: 4,746,441
  test rows:     901,509
Output shards:  16
  0.parquet  (282,074 rows)
  1.parquet  (310,160 rows)
  2.parquet  (349,538 rows)
  3.parquet  (383,700 rows)
  4.parquet  (430,680 rows)
  5.parquet  (331,578 rows)
  6.parquet  (443,003 rows)
  7.parquet  (380,309 rows)
  8.parquet  (349,822 rows)
  9.parquet  (299,581 rows)
  a.parquet  (411,142 rows)
  b.parquet  (347,906 rows)
  c.parquet  (340,277 rows)
  d.parquet  (353,024 rows)
  e.parquet  (299,764 rows)
  f.parquet  (335,392 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [72]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [73]:
df = pd.read_parquet(ENRICHED_TRADES_DIR/'enriched_3.parquet')
len(df)

4110058

In [74]:
len(df[(df['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')])

268526

In [75]:
dfs = []
for f in sorted(ENRICHED_TRADES_DIR.glob("*.parquet")):
    dfp = pd.read_parquet(f)
    dfs.append(dfp[dfp['wallet'] == '0x96489abcb9f583d6835c8ef95ffc923d05a86825'])

df_full = pd.concat(dfs, ignore_index=True)

In [76]:
pd.set_option('display.max_columns', None)

In [77]:
enriched = df_full.copy()
enriched["dt"] = pd.to_datetime(enriched["block_timestamp"], unit="s", utc=True)
enriched['final_price'] = enriched['token_winner'] * 1
enriched["final_value_usdc"] = enriched["quantity"] * enriched["final_price"]
enriched["price_x_qty"] = enriched["price"] * enriched["quantity"]

_GROUP_KEYS = ["tx_hash", "wallet", "side"]
# Group fills → one row per tx_hash × wallet × side
grouped = (
    enriched.groupby(_GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",              "first"),
        condition_id     = ("condition_id",    "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("quantity",         "sum"),
        price_x_qty_sum  = ("price_x_qty",     "sum"),
        trade_value_usdc = ("usdc_amount",      "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("log_index",        "count"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)

grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]

is_buy = grouped["side"] == "BUY"
grouped["trade_pnl"] = np.where(
    is_buy,
    grouped["final_value_usdc"] - grouped["trade_value_usdc"],
    grouped["trade_value_usdc"] - grouped["final_value_usdc"],
)

In [78]:
grouped.groupby('condition_id').agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum'),
    total_pnl    = ('trade_pnl', 'sum'),
).sort_values('total_pnl', ascending=True).head(10)

,total_trades,total_volume,total_pnl
condition_id,,,
0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,2838,1589594.90,-1589594.90
0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,1443,745147.28,-631003.04
0x4b02efe53e631ada84681303fd66d79ad615f3d2b6a28b4633d43d935f89af58,1555,585240.20,-577608.94
0xdc4b9ecbfac2c8e98fe18b35c4578a3c64e34ddfe16b1cf8a98ab3ad9d06c088,962,463369.04,-463369.04
0x15aa3c1259a716915e068a0d63c3885d2301d29e8982cbb1717ecb9b63d02d95,857,448182.82,-448182.82
0x70909f0ba8256a89c301da58812ae47203df54957a07c7f8b10235e877ad63c2,1256,477185.87,-419092.55
0x291f0f6023efa933d36cc80fd55f9d176309a92b61b9567fd4f044a8b21873fb,769,409423.70,-409423.70
0x1db02ba50e2312a62b4104de691cc7a76065d8d0da40decf93eb1b914a3217b7,726,577558.55,-347391.11
0x797d586ad45522306490b0cc9b2f21bdf957f3843476fae99f3bcc2cec83b74b,2524,336785.66,-334649.02


In [79]:
grouped.groupby(grouped['dt'].dt.date).agg(
    total_trades = ('num_fills', 'sum'),
    total_volume = ('trade_value_usdc', 'sum')
).sort_index(ascending=False).head(10)

,total_trades,total_volume
dt,,
2026-05-26,11,19517.50
2026-05-25,9,16314.82
2026-05-24,23,32065.62
2026-05-23,66,11930.73
2026-05-22,27,13951.33
2026-05-21,50,23981.54
2026-05-20,54,27128.40
2026-05-19,57,7815.94
2026-05-18,118,29976.49


In [80]:
df = pd.read_parquet('/Users/vobornij/projects/polymarket/data/polygon/2026-05-04.parquet')

In [81]:
df.tail()

,block_number,tx_hash,log_index,block_timestamp,order_hash,maker,taker,maker_asset_id,taker_asset_id,protocol_version,side,token_id,maker_amount_filled,taker_amount_filled,fee,builder,metadata
5989041,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5281,1777939198,0x7cd92feca2c9c2aa649cd0c0b34644b70f0ceba49ad9b6b9bd3c33f4685ad70c,0x106a48725a926362bbb137d4c8bccedbaa6aacee,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,10000900,10990000,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5989042,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5283,1777939198,0xf6d86802a56c0d0b6595bcb328576bb958732775eb5ceeac3aad1e6ec44a122f,0x8da42eb2e2c1eba87794e4c30bc689513b90f3d1,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,3649100,4010000,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5989043,86409577,0xcec46f9200e900d537f5b39e84b48690972c151c3de4774b5e43801cfea09f4d,5287,1777939198,0xce23af9a8098a0eb0e69f8ddea7fdd4921d52dd73ce6b3c59c5204a885a7c137,0x33ab02523163fc3e88ddaa381ca1fa05284606d9,0xe111180000d2663c0091e4f400237545b87b996b,0,28348355012943795476930758287039002184648591224808991117129985540950827926854,V2,BUY,28348355012943795476930758287039002184648591224808991117129985540950827926854,4752900,52810000,311410,0x4898df15ec6590495dc6c0fedf951ade3e64001d47f9caf44a64e86fc11959df,0x0000000000000000000000000000000000000000000000000000000000000000
5989044,86409577,0xd66008edc4f19dc239264ebc0687d2e7e85b5183670cd5e9e3e9da1af599bc66,5303,1777939198,0x81f68266bb8d09be6ffce910ea780b506d9ef2540bf34a965f2aaf048e9e38b7,0xae3db1cc155e1b2a5420e256c42cd1649b5a6b95,0x63adda169ca8f3311b361e3e7dc1a98fa882c072,0,28348355012943795476930758287039002184648591224808991117129985540950827926854,V2,BUY,28348355012943795476930758287039002184648591224808991117129985540950827926854,14338260,179228250,0,0x8ba7e2a23e33175c53fccfa30261a35734772d5909e4e4f484f204bfde446c7a,0x0000000000000000000000000000000000000000000000000000000000000000
5989045,86409577,0xd66008edc4f19dc239264ebc0687d2e7e85b5183670cd5e9e3e9da1af599bc66,5307,1777939198,0x012d6a71961ef6008f0bb1e8646c7b74b4a0eca2bf7ebb6358541c0d8f01287c,0x63adda169ca8f3311b361e3e7dc1a98fa882c072,0xe111180000d2663c0091e4f400237545b87b996b,0,20315904154374472650661835345885206083613913192457947093594900871269080896805,V2,BUY,20315904154374472650661835345885206083613913192457947093594900871269080896805,164889990,179228250,949760,0x9fedc0b0702ca6cb294e5321d9491b1e38b8bd2b463a7f7b06df8e6d7553cd18,0x0000000000000000000000000000000000000000000000000000000000000000
